# Employee Attrition



In [1]:
# IMPORTS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from scipy.stats import randint, uniform


import joblib

import warnings
warnings.filterwarnings("ignore")


In [4]:
# LOAD DATASET
df = pd.read_csv(
    r'C:\Users\nagaraj nalla\Downloads\Employee\Employee Attrition.csv'
)

print("Dataset Shape:", df.shape)
display(df.head())


Dataset Shape: (59598, 24)


,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,...,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,...,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,...,3,Mid,Medium,21,No,No,No,Fair,Low,Stayed
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,...,3,Mid,Medium,74,No,No,No,Poor,Low,Stayed
3,65791,36,Female,7,Education,3989,Good,High,High,1,...,2,Mid,Small,50,Yes,No,No,Good,Medium,Stayed
4,65026,56,Male,41,Education,4821,Fair,Very High,Average,0,...,0,Senior,Medium,68,No,No,No,Fair,Medium,Stayed


In [5]:
# TARGET ENCODING + FEATURES / TARGET
df['Attrition'] = df['Attrition'].map({
    'Stayed': 0,
    'Left': 1
})

X = df.drop(columns=['Attrition', 'Employee ID'])
y = df['Attrition']

print("X Shape:", X.shape)
print("y Shape:", y.shape)
print("\nTarget Distribution:")
print(y.value_counts())


X Shape: (59598, 22)
y Shape: (59598,)

Target Distribution:
Attrition
0    31260
1    28338
Name: count, dtype: int64


In [6]:
# TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train Shape:", X_train.shape)
print("X_test Shape :", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape :", y_test.shape)


X_train Shape: (47678, 22)
X_test Shape : (11920, 22)
y_train Shape: (47678,)
y_test Shape : (11920,)


In [7]:
# FEATURE GROUPS

numerical_cols = [
    'Age',
    'Years at Company',
    'Monthly Income',
    'Number of Promotions',
    'Distance from Home',
    'Number of Dependents',
    'Company Tenure'
]

nominal_cols = [
    'Gender',
    'Job Role',
    'Marital Status',
    'Remote Work',
    'Overtime',
    'Leadership Opportunities',
    'Innovation Opportunities'
]

ordinal_cols = [
    'Work-Life Balance',
    'Job Satisfaction',
    'Performance Rating',
    'Education Level',
    'Job Level',
    'Company Size',
    'Company Reputation',
    'Employee Recognition'
]

print("Numerical Columns :", numerical_cols)
print("Nominal Columns   :", nominal_cols)
print("Ordinal Columns   :", ordinal_cols)


Numerical Columns : ['Age', 'Years at Company', 'Monthly Income', 'Number of Promotions', 'Distance from Home', 'Number of Dependents', 'Company Tenure']
Nominal Columns   : ['Gender', 'Job Role', 'Marital Status', 'Remote Work', 'Overtime', 'Leadership Opportunities', 'Innovation Opportunities']
Ordinal Columns   : ['Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Education Level', 'Job Level', 'Company Size', 'Company Reputation', 'Employee Recognition']


In [8]:
# ORDINAL CATEGORY ORDER

ordinal_categories = [
    ['Poor', 'Fair', 'Good', 'Excellent'],                         # Work-Life Balance
    ['Low', 'Medium', 'High', 'Very High'],                       # Job Satisfaction
    ['Below Average', 'Low', 'Average', 'High'],                  # Performance Rating
    ['High School', 'Associate Degree', "Bachelor's Degree",
     "Master's Degree", 'PhD'],                                   # Education Level
    ['Entry', 'Mid', 'Senior'],                                   # Job Level
    ['Small', 'Medium', 'Large'],                                 # Company Size
    ['Poor', 'Fair', 'Good', 'Excellent'],                        # Company Reputation
    ['Low', 'Medium', 'High', 'Very High']                        # Employee Recognition
]


In [10]:
# PREPROCESSING PIPELINE


numerical_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

nominal_pipeline = Pipeline([
    ('onehot', OneHotEncoder(
        sparse_output=False,
        handle_unknown='ignore',
        drop='first'
    ))
])

ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_cols),
        ('nom', nominal_pipeline, nominal_cols),
        ('ord', ordinal_pipeline, ordinal_cols)
    ],
    remainder='drop'
)

print("Preprocessing pipeline created.")


Preprocessing pipeline created.


## KNN — Model Pipeline + Baseline Evaluation

## Model Pipelines — No Feature Selection



In [12]:
# KNN - PIPELINE + BASELINE EVALUATION

knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=3))
])

knn_pipeline.fit(X_train, y_train)

y_pred_knn = knn_pipeline.predict(X_test)

print("KNN BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_knn))
print("Precision :", precision_score(y_test, y_pred_knn))
print("Recall    :", recall_score(y_test, y_pred_knn))
print("F1 Score  :", f1_score(y_test, y_pred_knn))

if hasattr(knn_pipeline, "predict_proba"):
    y_prob_knn = knn_pipeline.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_knn))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))


KNN BASELINE RESULTS
Accuracy  : 0.661744966442953
Precision : 0.6413268832066344
Recall    : 0.6549047282992237
F1 Score  : 0.6480446927374302
ROC-AUC   : 0.7058405248217536

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.67      0.67      6252
           1       0.64      0.65      0.65      5668

    accuracy                           0.66     11920
   macro avg       0.66      0.66      0.66     11920
weighted avg       0.66      0.66      0.66     11920


Confusion Matrix:
[[4176 2076]
 [1956 3712]]


In [ ]:
knn_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Years at Company',
                                                   'Monthly Income',
                                                   'Number of Promotions',
                                                   'Distance from Home',
                                                   'Number of Dependents',
                                                   'Company Tenure']),
                                                 ('nom',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Gend...
                                                                                              ['Small',
                                                                                               'Medium',
                                                                                               'Large'],
                                                                                              ['Poor',
                                                                                               'Fair',
                                                                                               'Good',
                                                                                               'Excellent'],
                                                                                              ['Low',
                                                                                               'Medium',
                                                                                               'High',
                                                                                               'Very '
                                                                                               'High']],
                                                                                  handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['Work-Life Balance',
                                                   'Job Satisfaction',
                                                   'Performance Rating',
                                                   'Education Level',
                                                   'Job Level', 'Company Size',
                                                   'Company Reputation',
                                                   'Employee Recognition'])])),
                ('classifier', KNeighborsClassifier(n_neighbors=3))])

In [ ]:
# KNN - HYPERPARAMETER TUNING

param_grid_knn = {'classifier__n_neighbors': [3, 5, 7, 9, 11], 'classifier__weights': ['uniform', 'distance'], 'classifier__metric': ['euclidean', 'manhattan']}

grid_search_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid_knn,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_knn.fit(X_train, y_train)

best_knn_model = grid_search_knn.best_estimator_

print("KNN HYPERPARAMETER TUNING COMPLETED")
print("=" * 60)
print("Best Parameters:")
print(grid_search_knn.best_params_)
print("\nBest CV F1 Score:", grid_search_knn.best_score_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
KNN HYPERPARAMETER TUNING COMPLETED
Best Parameters:
{'classifier__metric': 'manhattan', 'classifier__n_neighbors': 11, 'classifier__weights': 'distance'}

Best CV F1 Score: 0.6957364195726716


In [ ]:
# KNN - FINAL TUNED EVALUATION

y_pred_best_knn = best_knn_model.predict(X_test)

print("TUNED KNN RESULTS")
print("=" * 60)

print("Best Parameters :", grid_search_knn.best_params_)
print("\nAccuracy  :", accuracy_score(y_test, y_pred_best_knn))
print("Precision :", precision_score(y_test, y_pred_best_knn))
print("Recall    :", recall_score(y_test, y_pred_best_knn))
print("F1 Score  :", f1_score(y_test, y_pred_best_knn))

if hasattr(best_knn_model, "predict_proba"):
    y_prob_best_knn = best_knn_model.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_best_knn))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_knn))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_knn))


TUNED KNN RESULTS
Best Parameters : {'classifier__metric': 'manhattan', 'classifier__n_neighbors': 11, 'classifier__weights': 'distance'}

Accuracy  : 0.7121644295302013
Precision : 0.6918195849768479
Recall    : 0.7117148906139732
F1 Score  : 0.7016262283676842
ROC-AUC   : 0.7826844739252952

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.71      0.72      6252
           1       0.69      0.71      0.70      5668

    accuracy                           0.71     11920
   macro avg       0.71      0.71      0.71     11920
weighted avg       0.71      0.71      0.71     11920


Confusion Matrix:
[[4455 1797]
 [1634 4034]]


## Decision Tree — Model Pipeline + Baseline Evaluation

In [ ]:
# DECISION TREE - PIPELINE + BASELINE EVALUATION

dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42))
])

dt_pipeline.fit(X_train, y_train)

y_pred_dt = dt_pipeline.predict(X_test)

print("DECISION TREE BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_dt))
print("Precision :", precision_score(y_test, y_pred_dt))
print("Recall    :", recall_score(y_test, y_pred_dt))
print("F1 Score  :", f1_score(y_test, y_pred_dt))

if hasattr(dt_pipeline, "predict_proba"):
    y_prob_dt = dt_pipeline.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))


DECISION TREE BASELINE RESULTS
Accuracy  : 0.7182885906040268
Precision : 0.6709085528262799
Recall    : 0.7999294283697953
F1 Score  : 0.7297601802671817
ROC-AUC   : 0.8018711923264301

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.64      0.71      6252
           1       0.67      0.80      0.73      5668

    accuracy                           0.72     11920
   macro avg       0.73      0.72      0.72     11920
weighted avg       0.73      0.72      0.72     11920


Confusion Matrix:
[[4028 2224]
 [1134 4534]]


In [ ]:

# DECISION TREE - HYPERPARAMETER TUNING

param_grid_dt = {'classifier__max_depth': [3, 5, 7, 10, 15, None],
                 'classifier__min_samples_split': [2, 5, 10, 20],
                 'classifier__min_samples_leaf': [1, 2, 5, 10],
                 'classifier__criterion': ['gini', 'entropy']}

grid_search_dt = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid_dt,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_dt.fit(X_train, y_train)

best_dt_model = grid_search_dt.best_estimator_

print("DECISION TREE HYPERPARAMETER TUNING COMPLETED")
print("=" * 60)
print("Best Parameters:")
print(grid_search_dt.best_params_)
print("\nBest CV F1 Score:", grid_search_dt.best_score_)


Fitting 5 folds for each of 192 candidates, totalling 960 fits
DECISION TREE HYPERPARAMETER TUNING COMPLETED
Best Parameters:
{'classifier__criterion': 'entropy', 'classifier__max_depth': 7, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 20}

Best CV F1 Score: 0.5907222231110378


In [ ]:
# DECISION TREE - PIPELINE + BASELINE EVALUATION

dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42))
])

dt_pipeline.fit(X_train, y_train)

y_pred_dt = dt_pipeline.predict(X_test)

print("DECISION TREE BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_dt))
print("Precision :", precision_score(y_test, y_pred_dt))
print("Recall    :", recall_score(y_test, y_pred_dt))
print("F1 Score  :", f1_score(y_test, y_pred_dt))

if hasattr(dt_pipeline, "predict_proba"):
    y_prob_dt = dt_pipeline.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))


DECISION TREE BASELINE RESULTS
Accuracy  : 0.7182885906040268
Precision : 0.6709085528262799
Recall    : 0.7999294283697953
F1 Score  : 0.7297601802671817
ROC-AUC   : 0.8018711923264301

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.64      0.71      6252
           1       0.67      0.80      0.73      5668

    accuracy                           0.72     11920
   macro avg       0.73      0.72      0.72     11920
weighted avg       0.73      0.72      0.72     11920


Confusion Matrix:
[[4028 2224]
 [1134 4534]]


In [ ]:
# DECISION TREE - RANDOMIZED HYPERPARAMETER TUNING

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions_dt = {
    "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],

    "classifier__min_samples_split": randint(2, 30),

    "classifier__min_samples_leaf": randint(1, 15),

    "classifier__criterion": ["gini", "entropy"]
}

random_search_dt = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=param_distributions_dt,
    n_iter=30,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_dt.fit(X_train, y_train)

best_dt_model = random_search_dt.best_estimator_

print("DECISION TREE RANDOMIZED SEARCH COMPLETED")
print("=" * 60)

print("Best Parameters:")
print(random_search_dt.best_params_)

print("\nBest CV F1 Score:")
print(random_search_dt.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
DECISION TREE RANDOMIZED SEARCH COMPLETED
Best Parameters:
{'classifier__criterion': 'entropy', 'classifier__max_depth': 7, 'classifier__min_samples_leaf': 12, 'classifier__min_samples_split': 24}

Best CV F1 Score:
0.7293485159722101


In [ ]:
# DECISION TREE - FINAL TUNED EVALUATION

y_pred_best_dt = best_dt_model.predict(X_test)

print("TUNED DECISION TREE RESULTS")
print("=" * 60)

print("Best Parameters :", random_search_dt.best_params_)

print("\nAccuracy  :", accuracy_score(y_test, y_pred_best_dt))
print("Precision :", precision_score(y_test, y_pred_best_dt))
print("Recall    :", recall_score(y_test, y_pred_best_dt))
print("F1 Score  :", f1_score(y_test, y_pred_best_dt))

if hasattr(best_dt_model, "predict_proba"):
    y_prob_best_dt = best_dt_model.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_best_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_dt))

TUNED DECISION TREE RESULTS
Best Parameters : {'classifier__criterion': 'entropy', 'classifier__max_depth': 7, 'classifier__min_samples_leaf': 12, 'classifier__min_samples_split': 24}

Accuracy  : 0.7323825503355704
Precision : 0.6973558458107677
Recall    : 0.7724064925899788
F1 Score  : 0.7329650092081031
ROC-AUC   : 0.8118351598201351

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.70      0.73      6252
           1       0.70      0.77      0.73      5668

    accuracy                           0.73     11920
   macro avg       0.73      0.73      0.73     11920
weighted avg       0.74      0.73      0.73     11920


Confusion Matrix:
[[4352 1900]
 [1290 4378]]


## Naive Bayes — Model Pipeline + Baseline Evaluation

In [ ]:
# NAIVE BAYES - PIPELINE + BASELINE EVALUATION


nb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB())
])

nb_pipeline.fit(X_train, y_train)

y_pred_nb = nb_pipeline.predict(X_test)

print("NAIVE BAYES BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_nb))
print("Precision :", precision_score(y_test, y_pred_nb))
print("Recall    :", recall_score(y_test, y_pred_nb))
print("F1 Score  :", f1_score(y_test, y_pred_nb))

if hasattr(nb_pipeline, "predict_proba"):
    y_prob_nb = nb_pipeline.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))


NAIVE BAYES BASELINE RESULTS
Accuracy  : 0.7337248322147651
Precision : 0.715893351800554
Recall    : 0.7295342272406493
F1 Score  : 0.722649423278574
ROC-AUC   : 0.8178781801820594

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.74      0.74      6252
           1       0.72      0.73      0.72      5668

    accuracy                           0.73     11920
   macro avg       0.73      0.73      0.73     11920
weighted avg       0.73      0.73      0.73     11920


Confusion Matrix:
[[4611 1641]
 [1533 4135]]


In [ ]:
# NAIVE BAYES - HYPERPARAMETER TUNING


param_grid_nb = {'classifier__var_smoothing': [1e-12,
                                               1e-11,
                                               1e-10,
                                               1e-09,
                                               1e-08,
                                               1e-07,
                                               1e-06,
                                               1e-05]}

grid_search_nb = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=param_grid_nb,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_nb.fit(X_train, y_train)

best_nb_model = grid_search_nb.best_estimator_

print("NAIVE BAYES HYPERPARAMETER TUNING COMPLETED")
print("=" * 60)
print("Best Parameters:")
print(grid_search_nb.best_params_)
print("\nBest CV F1 Score:", grid_search_nb.best_score_)


Fitting 5 folds for each of 8 candidates, totalling 40 fits
NAIVE BAYES HYPERPARAMETER TUNING COMPLETED
Best Parameters:
{'classifier__var_smoothing': 1e-12}

Best CV F1 Score: 0.7191907159361891


In [ ]:
# NAIVE BAYES - FINAL TUNED EVALUATION


y_pred_best_nb = best_nb_model.predict(X_test)

print("TUNED NAIVE BAYES RESULTS")
print("=" * 60)

print("Best Parameters :", grid_search_nb.best_params_)
print("\nAccuracy  :", accuracy_score(y_test, y_pred_best_nb))
print("Precision :", precision_score(y_test, y_pred_best_nb))
print("Recall    :", recall_score(y_test, y_pred_best_nb))
print("F1 Score  :", f1_score(y_test, y_pred_best_nb))

if hasattr(best_nb_model, "predict_proba"):
    y_prob_best_nb = best_nb_model.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_best_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_nb))


TUNED NAIVE BAYES RESULTS
Best Parameters : {'classifier__var_smoothing': 1e-12}

Accuracy  : 0.7337248322147651
Precision : 0.715893351800554
Recall    : 0.7295342272406493
F1 Score  : 0.722649423278574
ROC-AUC   : 0.8178781801820594

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.74      0.74      6252
           1       0.72      0.73      0.72      5668

    accuracy                           0.73     11920
   macro avg       0.73      0.73      0.73     11920
weighted avg       0.73      0.73      0.73     11920


Confusion Matrix:
[[4611 1641]
 [1533 4135]]


## Logistic Regression — Model Pipeline + Baseline Evaluation

In [ ]:
# LOGISTIC REGRESSION - PIPELINE + BASELINE EVALUATION


lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)

print("LOGISTIC REGRESSION BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_lr))
print("Precision :", precision_score(y_test, y_pred_lr))
print("Recall    :", recall_score(y_test, y_pred_lr))
print("F1 Score  :", f1_score(y_test, y_pred_lr))

if hasattr(lr_pipeline, "predict_proba"):
    y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))


LOGISTIC REGRESSION BASELINE RESULTS
Accuracy  : 0.7398489932885906
Precision : 0.7285841495992876
Recall    : 0.7217713479181369
F1 Score  : 0.7251617477621201
ROC-AUC   : 0.8309397167923906

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.76      0.75      6252
           1       0.73      0.72      0.73      5668

    accuracy                           0.74     11920
   macro avg       0.74      0.74      0.74     11920
weighted avg       0.74      0.74      0.74     11920


Confusion Matrix:
[[4728 1524]
 [1577 4091]]


In [ ]:
# LOGISTIC REGRESSION - HYPERPARAMETER TUNING


param_grid_lr = {'classifier__C': [0.01, 0.1, 1, 10, 100],
                 'classifier__penalty': ['l1', 'l2'],
                 'classifier__solver': ['liblinear']}

grid_search_lr = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid_lr,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_lr.fit(X_train, y_train)

best_lr_model = grid_search_lr.best_estimator_

print("LOGISTIC REGRESSION HYPERPARAMETER TUNING COMPLETED")
print("=" * 60)
print("Best Parameters:")
print(grid_search_lr.best_params_)
print("\nBest CV F1 Score:", grid_search_lr.best_score_)


Fitting 5 folds for each of 10 candidates, totalling 50 fits
LOGISTIC REGRESSION HYPERPARAMETER TUNING COMPLETED
Best Parameters:
{'classifier__C': 100, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}

Best CV F1 Score: 0.7301546335632834


In [ ]:
# LOGISTIC REGRESSION - FINAL TUNED EVALUATION


y_pred_best_lr = best_lr_model.predict(X_test)

print("TUNED LOGISTIC REGRESSION RESULTS")
print("=" * 60)

print("Best Parameters :", grid_search_lr.best_params_)
print("\nAccuracy  :", accuracy_score(y_test, y_pred_best_lr))
print("Precision :", precision_score(y_test, y_pred_best_lr))
print("Recall    :", recall_score(y_test, y_pred_best_lr))
print("F1 Score  :", f1_score(y_test, y_pred_best_lr))

if hasattr(best_lr_model, "predict_proba"):
    y_prob_best_lr = best_lr_model.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_best_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_lr))


TUNED LOGISTIC REGRESSION RESULTS
Best Parameters : {'classifier__C': 100, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}

Accuracy  : 0.7400167785234899
Precision : 0.7286807904575396
Recall    : 0.7221242060691602
F1 Score  : 0.725387682764732
ROC-AUC   : 0.8309352298725241

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.76      0.75      6252
           1       0.73      0.72      0.73      5668

    accuracy                           0.74     11920
   macro avg       0.74      0.74      0.74     11920
weighted avg       0.74      0.74      0.74     11920


Confusion Matrix:
[[4728 1524]
 [1575 4093]]


## SVC — Model Pipeline + Baseline Evaluation

In [8]:
# SVC - PIPELINE + BASELINE EVALUATION


svc_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(kernel='rbf', probability=True, random_state=42))
])

svc_pipeline.fit(X_train, y_train)

y_pred_svc = svc_pipeline.predict(X_test)

print("SVC BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_svc))
print("Precision :", precision_score(y_test, y_pred_svc))
print("Recall    :", recall_score(y_test, y_pred_svc))
print("F1 Score  :", f1_score(y_test, y_pred_svc))

if hasattr(svc_pipeline, "predict_proba"):
    y_prob_svc = svc_pipeline.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_svc))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svc))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svc))


SVC BASELINE RESULTS
Accuracy  : 0.7462248322147651
Precision : 0.7333568779798694
Recall    : 0.7327099505998589
F1 Score  : 0.7330332715559086
ROC-AUC   : 0.8406491291876225

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.76      0.76      6252
           1       0.73      0.73      0.73      5668

    accuracy                           0.75     11920
   macro avg       0.75      0.75      0.75     11920
weighted avg       0.75      0.75      0.75     11920


Confusion Matrix:
[[4742 1510]
 [1515 4153]]


In [9]:
svc_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Years at Company',
                                                   'Monthly Income',
                                                   'Number of Promotions',
                                                   'Distance from Home',
                                                   'Number of Dependents',
                                                   'Company Tenure']),
                                                 ('nom',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Gend...
                                                                                              ['Small',
                                                                                               'Medium',
                                                                                               'Large'],
                                                                                              ['Poor',
                                                                                               'Fair',
                                                                                               'Good',
                                                                                               'Excellent'],
                                                                                              ['Low',
                                                                                               'Medium',
                                                                                               'High',
                                                                                               'Very '
                                                                                               'High']],
                                                                                  handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['Work-Life Balance',
                                                   'Job Satisfaction',
                                                   'Performance Rating',
                                                   'Education Level',
                                                   'Job Level', 'Company Size',
                                                   'Company Reputation',
                                                   'Employee Recognition'])])),
                ('classifier', SVC(probability=True, random_state=42))])

In [11]:
# SVC - RANDOMIZED HYPERPARAMETER TUNING

from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from scipy.stats import loguniform

# 1. SAMPLE TRAINING DATA FOR TUNING
X_tune_svc = X_train.sample(
    n=min(30000, len(X_train)),
    random_state=42
)

y_tune_svc = y_train.loc[X_tune_svc.index]


# 2. SVC PIPELINE
svc_tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        random_state=42
    ))
])


# 3. SMALL HYPERPARAMETER SEARCH
param_dist_svc = {
    "classifier__C": loguniform(0.1, 10),
    "classifier__kernel": ["rbf", "linear"],
    "classifier__gamma": ["scale", "auto"]
}


# 4. RANDOMIZED SEARCH
random_search_svc = RandomizedSearchCV(
    estimator=svc_tuning_pipeline,
    param_distributions=param_dist_svc,
    n_iter=3,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)



# 5. FIT ONLY ON SAMPLE
random_search_svc.fit(
    X_tune_svc,
    y_tune_svc
)


# 6. BEST PARAMETERS
print("SVC RANDOMIZED SEARCH COMPLETED")
print("=" * 60)

print("Best Parameters:")
print(random_search_svc.best_params_)

print("\nBest CV F1 Score:")
print(random_search_svc.best_score_)

Fitting 3 folds for each of 3 candidates, totalling 9 fits
SVC RANDOMIZED SEARCH COMPLETED
Best Parameters:
{'classifier__C': np.float64(0.20513382630874505), 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}

Best CV F1 Score:
0.733999495700575


In [14]:
# SVC - FINAL TUNED EVALUATION

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# 1. BEST SVC MODEL
best_svc_model = random_search_svc.best_estimator_

# 2. PREDICTIONS
y_pred_best_svc = best_svc_model.predict(X_test)



# 3. EVALUATION
print("TUNED SVC RESULTS")
print("=" * 60)

print("Best Parameters :", random_search_svc.best_params_)

print("\nAccuracy  :", accuracy_score(y_test, y_pred_best_svc))
print("Precision :", precision_score(y_test, y_pred_best_svc))
print("Recall    :", recall_score(y_test, y_pred_best_svc))
print("F1 Score  :", f1_score(y_test, y_pred_best_svc))


# 4. ROC-AUC
y_score_best_svc = best_svc_model.decision_function(X_test)

print("ROC-AUC   :", roc_auc_score(y_test, y_score_best_svc))



# 5. CLASSIFICATION REPORT
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_svc))



# 6. CONFUSION MATRIX
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_svc))

TUNED SVC RESULTS
Best Parameters : {'classifier__C': np.float64(0.20513382630874505), 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}

Accuracy  : 0.7458892617449664
Precision : 0.7377049180327869
Recall    : 0.7224770642201835
F1 Score  : 0.7300115874855156
ROC-AUC   : 0.8390636943954928

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.77      0.76      6252
           1       0.74      0.72      0.73      5668

    accuracy                           0.75     11920
   macro avg       0.75      0.74      0.75     11920
weighted avg       0.75      0.75      0.75     11920


Confusion Matrix:
[[4796 1456]
 [1573 4095]]


## Random Forest — Model Pipeline + Baseline Evaluation

In [15]:
# RANDOM FOREST - PIPELINE + BASELINE EVALUATION


rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)

print("RANDOM FOREST BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_rf))
print("Precision :", precision_score(y_test, y_pred_rf))
print("Recall    :", recall_score(y_test, y_pred_rf))
print("F1 Score  :", f1_score(y_test, y_pred_rf))

if hasattr(rf_pipeline, "predict_proba"):
    y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]
    print("ROC-AUC   :", roc_auc_score(y_test, y_prob_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))


RANDOM FOREST BASELINE RESULTS
Accuracy  : 0.7407718120805369
Precision : 0.7333454018826937
Recall    : 0.7147141848976711
F1 Score  : 0.7239099356683345
ROC-AUC   : 0.835235039536819

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.76      0.76      6252
           1       0.73      0.71      0.72      5668

    accuracy                           0.74     11920
   macro avg       0.74      0.74      0.74     11920
weighted avg       0.74      0.74      0.74     11920


Confusion Matrix:
[[4779 1473]
 [1617 4051]]


In [16]:
rf_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Years at Company',
                                                   'Monthly Income',
                                                   'Number of Promotions',
                                                   'Distance from Home',
                                                   'Number of Dependents',
                                                   'Company Tenure']),
                                                 ('nom',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Gend...
                                                                                               'Medium',
                                                                                               'Large'],
                                                                                              ['Poor',
                                                                                               'Fair',
                                                                                               'Good',
                                                                                               'Excellent'],
                                                                                              ['Low',
                                                                                               'Medium',
                                                                                               'High',
                                                                                               'Very '
                                                                                               'High']],
                                                                                  handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['Work-Life Balance',
                                                   'Job Satisfaction',
                                                   'Performance Rating',
                                                   'Education Level',
                                                   'Job Level', 'Company Size',
                                                   'Company Reputation',
                                                   'Employee Recognition'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [17]:
# RANDOM FOREST - RANDOMIZED HYPERPARAMETER TUNING

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist_rf = {
    "classifier__n_estimators": randint(100, 500),

    "classifier__max_depth": [10, 20, 30, 40, None],

    "classifier__min_samples_split": randint(2, 11),

    "classifier__min_samples_leaf": randint(1, 6),

    "classifier__max_features": ["sqrt", "log2"]
}

random_search_rf = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist_rf,
    n_iter=20,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search_rf.fit(X_train, y_train)

best_rf_model = random_search_rf.best_estimator_

print("RANDOM FOREST RANDOMIZED SEARCH COMPLETED")
print("=" * 60)

print("Best Parameters:")
print(random_search_rf.best_params_)

print("\nBest CV F1 Score:",
      random_search_rf.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
RANDOM FOREST RANDOMIZED SEARCH COMPLETED
Best Parameters:
{'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 466}

Best CV F1 Score: 0.7368516966288451


In [18]:
# RANDOM FOREST - FINAL TUNED EVALUATION


y_pred_best_rf = best_rf_model.predict(X_test)

print("TUNED RANDOM FOREST RESULTS")
print("=" * 60)

print("Best Parameters :",
      random_search_rf.best_params_)

print("\nAccuracy  :",
      accuracy_score(y_test, y_pred_best_rf))

print("Precision :",
      precision_score(y_test, y_pred_best_rf))

print("Recall    :",
      recall_score(y_test, y_pred_best_rf))

print("F1 Score  :",
      f1_score(y_test, y_pred_best_rf))

if hasattr(best_rf_model, "predict_proba"):
    y_prob_best_rf = best_rf_model.predict_proba(X_test)[:, 1]

    print("ROC-AUC   :",
          roc_auc_score(y_test, y_prob_best_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_rf))

TUNED RANDOM FOREST RESULTS
Best Parameters : {'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 466}

Accuracy  : 0.749496644295302
Precision : 0.7424078091106291
Recall    : 0.7245942131263232
F1 Score  : 0.7333928571428572
ROC-AUC   : 0.842568458544924

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.77      0.76      6252
           1       0.74      0.72      0.73      5668

    accuracy                           0.75     11920
   macro avg       0.75      0.75      0.75     11920
weighted avg       0.75      0.75      0.75     11920


Confusion Matrix:
[[4827 1425]
 [1561 4107]]


## XGBoost — Model Pipeline + Baseline Evaluation

In [13]:
# XGBOOST - PIPELINE + BASELINE EVALUATION

from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_pipeline.fit(X_train, y_train)

y_pred_xgb = xgb_pipeline.predict(X_test)
y_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

print("XGBOOST BASELINE RESULTS")
print("=" * 60)

print("Accuracy  :", accuracy_score(y_test, y_pred_xgb))
print("Precision :", precision_score(y_test, y_pred_xgb))
print("Recall    :", recall_score(y_test, y_pred_xgb))
print("F1 Score  :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC   :", roc_auc_score(y_test, y_prob_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))


XGBOOST BASELINE RESULTS
Accuracy  : 0.7421979865771812
Precision : 0.7282321899736148
Recall    : 0.7304163726182075
F1 Score  : 0.7293226459966529
ROC-AUC   : 0.8368973304689288

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.75      0.75      6252
           1       0.73      0.73      0.73      5668

    accuracy                           0.74     11920
   macro avg       0.74      0.74      0.74     11920
weighted avg       0.74      0.74      0.74     11920


Confusion Matrix:
[[4707 1545]
 [1528 4140]]


In [14]:
xgb_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['Age','Gender','Years at Company',...,'Innovation Opportunities', 'Company Reputation','Employee Recognition']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'`

In [15]:
# XGBOOST - RANDOMIZED HYPERPARAMETER TUNING
param_dist_xgb = {
    "classifier__n_estimators": randint(100, 500),
    "classifier__max_depth": randint(3, 10),
    "classifier__learning_rate": uniform(0.01, 0.19),
    "classifier__subsample": uniform(0.7, 0.3),
    "classifier__colsample_bytree": uniform(0.7, 0.3),
    "classifier__min_child_weight": randint(1, 10),
    "classifier__gamma": uniform(0, 0.5)
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search_xgb.fit(X_train, y_train)

best_xgb_model = random_search_xgb.best_estimator_

print("XGBOOST RANDOMIZED SEARCH COMPLETED")
print("=" * 60)
print("Best Parameters:")
print(random_search_xgb.best_params_)
print("\nBest CV F1 Score:", random_search_xgb.best_score_)


Fitting 3 folds for each of 20 candidates, totalling 60 fits
XGBOOST RANDOMIZED SEARCH COMPLETED
Best Parameters:
{'classifier__colsample_bytree': np.float64(0.8873062144401379), 'classifier__gamma': np.float64(0.147816842918857), 'classifier__learning_rate': np.float64(0.030043909367751413), 'classifier__max_depth': 6, 'classifier__min_child_weight': 4, 'classifier__n_estimators': 385, 'classifier__subsample': np.float64(0.9649840776756604)}

Best CV F1 Score: 0.7436314278314601


In [16]:
# XGBOOST - FINAL TUNED EVALUATION


y_pred_best_xgb = best_xgb_model.predict(X_test)
y_prob_best_xgb = best_xgb_model.predict_proba(X_test)[:, 1]

print("TUNED XGBOOST RESULTS")
print("=" * 60)

print("Best Parameters :", random_search_xgb.best_params_)
print("\nAccuracy  :", accuracy_score(y_test, y_pred_best_xgb))
print("Precision :", precision_score(y_test, y_pred_best_xgb))
print("Recall    :", recall_score(y_test, y_pred_best_xgb))
print("F1 Score  :", f1_score(y_test, y_pred_best_xgb))
print("ROC-AUC   :", roc_auc_score(y_test, y_prob_best_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_xgb))


TUNED XGBOOST RESULTS
Best Parameters : {'classifier__colsample_bytree': np.float64(0.8873062144401379), 'classifier__gamma': np.float64(0.147816842918857), 'classifier__learning_rate': np.float64(0.030043909367751413), 'classifier__max_depth': 6, 'classifier__min_child_weight': 4, 'classifier__n_estimators': 385, 'classifier__subsample': np.float64(0.9649840776756604)}

Accuracy  : 0.7546140939597316
Precision : 0.7414187643020596
Recall    : 0.7431192660550459
F1 Score  : 0.7422680412371134
ROC-AUC   : 0.8485098036094928

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.77      0.77      6252
           1       0.74      0.74      0.74      5668

    accuracy                           0.75     11920
   macro avg       0.75      0.75      0.75     11920
weighted avg       0.75      0.75      0.75     11920


Confusion Matrix:
[[4783 1469]
 [1456 4212]]


## Final Model Comparison

In [ ]:
# SAVE DEPLOYMENT PIPELINE

# XGBoost is used here as the final deployment pipeline,


import pickle

with open('model.pkl','wb') as f :
    pickle.dump(best_xgb_model,f)